In [17]:
import numpy as np
import os
import pandas as pd
import re
from pathlib import Path

In [18]:
import pyreadr

In [19]:
print("yes im changing!"
)

yes im changing!


In [20]:
print(os.getcwd())

/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/notebooks


In [21]:
result = pyreadr.read_r("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data/obs.new.220040700.rda")

In [22]:

da = result["obs"]

print("min:", float(da.min(skipna=True)))
print("max:", float(da.max(skipna=True)))
print("mean:", float(da.mean(skipna=True)))
print("std:", float(da.std(skipna=True)))

qs = da.quantile([0, 0.01, 0.1, 0.5, 0.9, 0.99, 1.0], skipna=True)
print(qs)

min: 0.0
max: 1.7204732762442694
mean: 0.025912389260554395
std: 0.08981992061963555
<xarray.DataArray (quantile: 7)> Size: 56B
array([0.        , 0.        , 0.        , 0.        , 0.06550781,
       0.47773685, 1.72047328])
Coordinates:
  * quantile  (quantile) float64 56B 0.0 0.01 0.1 0.5 0.9 0.99 1.0


In [23]:
from __future__ import annotations

import numpy as np
import pandas as pd
import xarray as xr


def season_from_month(month: int) -> str:
    # DJF/MAM/JJA/SON
    if month in (12, 1, 2):
        return "DJF"
    if month in (3, 4, 5):
        return "MAM"
    if month in (6, 7, 8):
        return "JJA"
    return "SON"

def expand_event_to_samples_np(arr, event_id, lead_steps=3, hist_steps=3, step=1):
    if arr.ndim != 3:
        raise ValueError(f"Expected (H,W,T), got {arr.shape}")
    
    ### want to craate training pairs of X = [t0-20,t0 -10,t0], Y = [t0+10, t0+20, t0+30]
    
    H, W, T = arr.shape
    last = T - 1

    t0_min = hist_steps - 1          # minimum t0 so that t0-2, t0-1, t0 all exist
    t0_max = last - lead_steps       # maximum t0 so that t0+1, t0+2, t0+3 all exist

    if t0_max < t0_min:
        return (
            np.empty((0, hist_steps, H, W), dtype=np.float32),
            np.empty((0, lead_steps, H, W), dtype=np.float32),
            pd.DataFrame(columns=["event_id", "t0_idx", "target_idx_10", "target_idx_20", "target_idx_30"])
        )

    t0s = np.arange(t0_min, t0_max + 1, step, dtype=int) # get all the t0
    N = len(t0s)

    X = np.empty((N, hist_steps, H, W), dtype=np.float32) # 3 steps for motion training
    Y = np.empty((N, lead_steps, H, W), dtype=np.float32) # targets: +10, +20, +30 min

    #sliding window
    for i, t0 in enumerate(t0s):
        x = arr[:, :, (t0 - (hist_steps - 1)):(t0 + 1)].to_numpy()
        X[i] = np.transpose(x, (2, 0, 1)).astype(np.float32, copy=False)

        for j in range(lead_steps):
            y = arr[:, :, t0 + j + 1].to_numpy()
            Y[i, j] = y.astype(np.float32, copy=False)

    # store metadata
    meta = pd.DataFrame({
    "event_id":      event_id,
    "t0_idx":        t0s,
    "target_idx_10": t0s + 1,    # +10min
    "target_idx_20": t0s + 2,    # +20min
    "target_idx_30": t0s + 3,    # +30min
    })

    return X, Y, meta


In [24]:
# Pick the observations array #

arr = result["obs"]

# Infer event_id from filename
path = "/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/obs.new.220040700.rda"
m = re.search(r"obs\.new\.(.+?)\.rda$", path)
event_id = m.group(1) if m else "unknown"

# Expand
X, Y, meta = expand_event_to_samples_np(arr, event_id=event_id, lead_steps=3, hist_steps=3, step=1)

print("Expanded samples:")
print("X:", X.shape)   # (N, 3, H, W)
print("Y:", Y.shape)   # (N, 3, H, W)
print(meta.head())

# Sanity: last indices
if len(meta) > 0:
    print("t0 range:", meta.t0_idx.min(), "..", meta.t0_idx.max())
    print("target_30 max:", meta.target_idx_30.max(), " / last index:", arr.shape[2] - 1)


Expanded samples:
X: (34, 3, 501, 371)
Y: (34, 3, 501, 371)
    event_id  t0_idx  target_idx_10  target_idx_20  target_idx_30
0  220040700       2              3              4              5
1  220040700       3              4              5              6
2  220040700       4              5              6              7
3  220040700       5              6              7              8
4  220040700       6              7              8              9
t0 range: 2 .. 35
target_30 max: 38  / last index: 38


In [25]:
print(meta.head(3))
print(meta.tail(3))
print(meta["t0_idx"].describe())
print(meta["t0_idx"].unique()[:10], " ... n_unique =", meta["t0_idx"].nunique())

    event_id  t0_idx  target_idx_10  target_idx_20  target_idx_30
0  220040700       2              3              4              5
1  220040700       3              4              5              6
2  220040700       4              5              6              7
     event_id  t0_idx  target_idx_10  target_idx_20  target_idx_30
31  220040700      33             34             35             36
32  220040700      34             35             36             37
33  220040700      35             36             37             38
count    34.000000
mean     18.500000
std       9.958246
min       2.000000
25%      10.250000
50%      18.500000
75%      26.750000
max      35.000000
Name: t0_idx, dtype: float64
[ 2  3  4  5  6  7  8  9 10 11]  ... n_unique = 34


In [26]:
from datetime import datetime,timedelta

def parse_event_datetime(event_id: str) -> datetime:
    """
    event_id format: YYDDDHHMM
    Example: '200500700' -> 2020, DOY 050, 07:00
    """
    s = str(event_id)
    if len(s) != 9:
        raise ValueError(f"Expected 9-digit YYDDDHHMM, got '{s}' (len={len(s)})")

    yy   = int(s[0:2])      # 20
    doy  = int(s[2:5])      # 050
    hhmm = s[5:9]           # 0700
    hour = int(hhmm[0:2])   # 07
    minute = int(hhmm[2:4]) # 00

    year = 2000 + yy
    return datetime.strptime(f"{year} {doy} {hour:02d} {minute:02d}", "%Y %j %H %M")


In [27]:
##### Loop over more years ######

# ------------------------------------------------
# -------------------- CONFIG --------------------
# ------------------------------------------------


DATA_DIR = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data")
FILES = sorted(DATA_DIR.glob("obs.new.*.rda"))

LEAD_STEPS = 3   # +30 min for 10-min timestep
HIST_STEPS = 3   # t0-20, t0-10, t0
STEP = 1         # 10-min stride (use 2 for 20-min stride)

R_ARRAY_KEY = "obs"
# ------------------------------------------------

print(f"Found {len(FILES)} files in {DATA_DIR}")

X_list, Y_list, meta_list = [], [], []
skipped = 0

for idx, f in enumerate(FILES, start=1):
    event_id = f.stem.split(".")[-1]  # obs.new.220040700 -> 220040700
    result = pyreadr.read_r(str(f))

    # Choose the array
    if R_ARRAY_KEY is not None:
        if R_ARRAY_KEY not in result:
            print(f"[{idx}/{len(FILES)}] Skip {f.name}: missing key {R_ARRAY_KEY}")
            skipped += 1
            continue
        arr = result[R_ARRAY_KEY]
        key_used = R_ARRAY_KEY
    else:
        candidates = [(k, v) for k, v in result.items() if hasattr(v, "ndim") and v.ndim == 3]
        if not candidates:
            print(f"[{idx}/{len(FILES)}] Skip {f.name}: no 3D array in keys={list(result.keys())}")
            skipped += 1
            continue
        key_used, arr = candidates[0]

    # Expand samples for this event
    X, Y, meta = expand_event_to_samples_np(
        arr, event_id=event_id, lead_steps=LEAD_STEPS, hist_steps=HIST_STEPS, step=STEP
    )

    if len(meta) == 0:
        print(f"[{idx}/{len(FILES)}] Skip {f.name}: not enough timesteps (arr shape={np.asarray(arr).shape})")
        skipped += 1
        continue

    # ---- Add more data artifacts ----
    event_dt = parse_event_datetime(event_id)

    meta["event_datetime"] = event_dt
    meta["year"] = event_dt.year
    meta["season"] = season_from_month(event_dt.month)

    dt_step = timedelta(minutes=10)

    meta["t0_datetime"]        = event_dt + meta["t0_idx"]        * dt_step
    meta["target_datetime_10"] = event_dt + meta["target_idx_10"] * dt_step
    meta["target_datetime_20"] = event_dt + meta["target_idx_20"] * dt_step
    meta["target_datetime_30"] = event_dt + meta["target_idx_30"] * dt_step

    X_list.append(X)
    Y_list.append(Y)
    meta_list.append(meta)

    if idx % 25 == 0 or idx == 1:
        print(f"[{idx}/{len(FILES)}] {f.name} key={key_used} arr={np.asarray(arr).shape} -> samples={len(meta)}")


Found 253 files in /store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data
[1/253] obs.new.200500700.rda key=obs arr=(501, 371, 39) -> samples=34
[25/253] obs.new.202061300.rda key=obs arr=(501, 371, 39) -> samples=34
[50/253] obs.new.203411600.rda key=obs arr=(501, 371, 39) -> samples=34
[75/253] obs.new.211242200.rda key=obs arr=(501, 371, 39) -> samples=34
[100/253] obs.new.211960400.rda key=obs arr=(501, 371, 39) -> samples=34
[125/253] obs.new.213611900.rda key=obs arr=(501, 371, 39) -> samples=34
[150/253] obs.new.221550400.rda key=obs arr=(501, 371, 39) -> samples=34
[175/253] obs.new.222671300.rda key=obs arr=(501, 371, 39) -> samples=34
[200/253] obs.new.230090100.rda key=obs arr=(501, 371, 39) -> samples=34
[225/253] obs.new.231211600.rda key=obs arr=(501, 371, 39) -> samples=34
[250/253] obs.new.232610100.rda key=obs arr=(501, 371, 39) -> samples=34


In [28]:
meta_all = pd.concat(meta_list, ignore_index=True) if meta_list else pd.DataFrame()
print("meta_all:", meta_all.shape)
print("Unique events:", meta_all["event_id"].nunique() if not meta_all.empty else 0)
out_csv = DATA_DIR / "training_samples_meta.csv"
meta_all.to_csv(out_csv, index=False)
#print("Saved:", out_csv)

meta_all: (8602, 12)
Unique events: 253


In [29]:
# Concatenate
X_all = np.concatenate(X_list, axis=0) if X_list else np.empty((0, HIST_STEPS, 0, 0), dtype=np.float32)
Y_all = np.concatenate(Y_list, axis=0) if Y_list else np.empty((0, LEAD_STEPS, 0, 0), dtype=np.float32)

print("\nDONE")
print("Skipped:", skipped)
print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)

X_all = np.nan_to_num(X_all, nan=0.0)
Y_all = np.nan_to_num(Y_all, nan=0.0)

# Save
out_npz = DATA_DIR / "training_data.npz"
np.savez_compressed(out_npz, X=X_all, Y=Y_all)
print("Saved:", out_npz)



DONE
Skipped: 0
X_all: (8602, 3, 501, 371)
Y_all: (8602, 3, 501, 371)
Saved: /store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data/training_data.npz
